In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from src.analysis.isc import (  # noqa: E402
    compute_loo_isc,
    compute_loo_isc_spearman,
    compute_mean_field_loo_isc,
    compute_mean_field_pairwise_isc,
    compute_mean_field_sliding_window_isc,
    compute_pairwise_isc,
    compute_pairwise_isc_spearman,
    compute_sliding_window_isc,
    compute_sliding_window_isc_spearman,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)
from src.visualization.isc_plots import (  # noqa: E402
    plot_loo_isc_pearson_vs_spearman,
    plot_mean_field_loo_isc,
    plot_mean_field_pairwise_isc,
    plot_mean_field_vs_channel_avg_isc,
    plot_multiscale_sliding_window_isc,
    plot_pairwise_isc_pearson_vs_spearman,
    print_data_overview,
)
from scripts.analysis_common import analyzers_to_datasets, load_analyzers  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# EEG Inter-Subject Correlation Analysis — Broadband (Raw, Unnormalized)

This notebook computes **inter-subject correlation (ISC)** on the raw EEG
signal (unnormalized amplitudes) across frequency bands:

1. **LOO-ISC distribution** — for each subject, Pearson and Spearman correlation
   against the mean of all others; histogram and per-subject violin plot
2. **Pairwise ISC matrix** — symmetric subject × subject correlation matrix;
   per-subject mean off-diagonal for outlier detection
3. **Sliding-window (time-resolved) ISC** — LOO-ISC as a time course with
   per-channel heatmap; Pearson vs Spearman comparison at medium window

Spearman functions (`compute_loo_isc_spearman`, `compute_pairwise_isc_spearman`,
`compute_sliding_window_isc_spearman`) are imported from `src.analysis.isc`.


## Configuration


In [ ]:
# ── Experiment configuration ───────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Data processing flag ───────────────────────────────────────────────────
# Set True to load raw EDF files, resample, stack, and save before analysis.
# Keep False to use already-saved concatenated arrays.
process_and_save_data = False

# ── LOO-ISC distribution parameters ───────────────────────────────────────
PLOT_PCT = 99  # clip histogram at this percentile to suppress outliers

# ── Sliding-window ISC parameters ─────────────────────────────────────────
# Three window sizes with 50% overlap (step = window / 2) for the sliding-window analyses.
# Fine   — appears nearly continuous; also used for the per-channel heatmap.
# Medium — main analysis window for the bar chart and Pearson/Spearman comparison.
# Large  — coarse long-range trend.
WINDOW_FINE_SEC = 1.0  # fine / one-step window (seconds)
WINDOW_MED_SEC = 5.0  # medium window (seconds)
WINDOW_LARGE_SEC = 15.0  # large window (seconds)
ISC_THRESHOLD = 0.035  # significance highlight threshold (r)

# Sub-sample channels for Spearman to keep runtime manageable during exploration.
# Set to None to use all channels (much slower).
N_CH_SUBSAMPLE = 64

# ── Plot saving ────────────────────────────────────────────────────────────
SAVE_PLOTS = True

PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "02-isc-broadband-analysis" / "plots" / "broadband"
)
print(f"Plots will be saved to: {PLOTS_DIR}")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading


In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type.
# normalize_data=False: keep raw amplitudes for this ISC analysis.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)
print_data_overview(datasets)

## Dataset Selection

Change `LABEL` to switch between music types. All analysis cells below use
`ad`, `data`, `n_subjects`, `n_channels`, `n_times`, and `sfreq`.


In [ ]:
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[f"{CONDITION.value}_{LABEL}"]
data = ad.data  # (n_subjects, n_channels, n_times) — unnormalized
sfreq = ad.sfreq

n_subjects, n_channels, n_times = data.shape
time = np.arange(n_times) / sfreq  # seconds

print(f"Dataset : {LABEL}")
print(f"Shape   : {data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print("Data: unnormalized (raw amplitude).")

## Section 1 — LOO-ISC Distribution (Pearson vs. Spearman)

For each subject *s*, the channel-wise correlation between subject *s* and the
**mean of all other subjects** is computed (**leave-one-out ISC**).

- **Histogram** — distribution of channel-wise mean LOO-ISC; how many channels
  show above-chance inter-subject agreement (clipped at `PLOT_PCT` to suppress outliers)
- **Per-subject violin + strip** — each dot is one subject's mean LOO-ISC
  averaged over channels; reveals outlier subjects
- **Pearson** captures linear synchrony; **Spearman** is robust to the amplitude
  outliers present


In [ ]:
print("Computing Pearson LOO-ISC …")
loo_pearson, mean_isc_pearson = compute_loo_isc(data)
# loo_pearson:      (n_subjects, n_channels)
# mean_isc_pearson: (n_channels,) — channel-wise group ISC

print("Computing Spearman LOO-ISC …")
loo_spearman, mean_isc_spearman = compute_loo_isc_spearman(data)

plot_loo_isc_pearson_vs_spearman(
    LABEL,
    loo_pearson,
    mean_isc_pearson,
    loo_spearman,
    mean_isc_spearman,
    plot_pct=PLOT_PCT,
    save_path_hist=PLOTS_DIR / f"loo_isc_distribution_{LABEL}.png"
    if SAVE_PLOTS
    else None,
    save_path_violin=PLOTS_DIR / f"loo_isc_per_subject_{LABEL}.png"
    if SAVE_PLOTS
    else None,
)

## Section 2 — Pairwise ISC Matrix (Pearson vs. Spearman)

Every pair of subjects is correlated (mean over channels) producing a symmetric
`(n_subjects × n_subjects)` matrix:

- **Heatmaps** — diagonal masked so the colour scale focuses on off-diagonal spread
- **Per-subject mean off-diagonal bar chart** — subjects with consistently low
  values are candidates for quality review
- **Off-diagonal distribution** — histogram comparing the spread of pairwise
  correlations for both methods


In [ ]:
print("Computing Pearson pairwise ISC …")
pair_pearson = compute_pairwise_isc(data)
# pair_pearson: (n_subjects, n_subjects)

print("Computing Spearman pairwise ISC …")
pair_spearman = compute_pairwise_isc_spearman(data)

plot_pairwise_isc_pearson_vs_spearman(
    LABEL,
    pair_pearson,
    pair_spearman,
    save_path_heatmaps=PLOTS_DIR / f"pairwise_isc_matrix_{LABEL}.png"
    if SAVE_PLOTS
    else None,
    save_path_per_subject=PLOTS_DIR / f"pairwise_isc_per_subject_{LABEL}.png"
    if SAVE_PLOTS
    else None,
    save_path_distribution=PLOTS_DIR / f"pairwise_isc_distribution_{LABEL}.png"
    if SAVE_PLOTS
    else None,
)

## Section 3 — Time-Resolved LOO-ISC (Multi-Scale Windowed Analysis)

50%-overlapping LOO-ISC is computed at **three temporal scales** (fine / medium / large, step = window / 2).

**Figure A — Bar chart (medium window, Pearson)**
Each bar is one 50%-overlapping window coloured green when its mean ISC exceeds `ISC_THRESHOLD`; error bars show ±std across channels. Highlights which segments of the recording have significant inter-subject synchrony.

**Figure B — Stair-step overlay + per-channel heatmap (Pearson)**
Upper panel overlays the three stair-step mean-ISC curves (fine / medium / large). Gold shading marks significant medium-window (50%-overlapping) spans. Lower panel is a fine-window per-channel ISC heatmap.

**Figure C — Pearson vs. Spearman comparison (medium window, 50% overlap)**
Overlay of Pearson and Spearman time courses showing how robust the synchrony signal is to rank-based estimation.


In [ ]:
# ── Channel subsampling (same subset for all methods / window sizes) ────────
if N_CH_SUBSAMPLE is not None and N_CH_SUBSAMPLE < n_channels:
    rng = np.random.default_rng(42)
    ch_idx = np.sort(rng.choice(n_channels, N_CH_SUBSAMPLE, replace=False))
    data_sw = data[:, ch_idx, :]
    print(f"Using {N_CH_SUBSAMPLE}/{n_channels} randomly subsampled channels.")
else:
    data_sw = data

# ── Compute LOO-ISC at 3 window sizes (Pearson) ─────────────────────────────
print(f"  Pearson ISC  ({WINDOW_FINE_SEC:.0f} s / 50% overlap) …")
isc_fine, times_fine = compute_sliding_window_isc(
    data_sw, WINDOW_FINE_SEC, WINDOW_FINE_SEC / 2, sfreq
)

print(f"  Pearson ISC  ({WINDOW_MED_SEC:.0f} s / 50% overlap) …")
isc_med, times_med = compute_sliding_window_isc(
    data_sw, WINDOW_MED_SEC, WINDOW_MED_SEC / 2, sfreq
)

print(f"  Pearson ISC  ({WINDOW_LARGE_SEC:.0f} s / 50% overlap) …")
isc_large, times_large = compute_sliding_window_isc(
    data_sw, WINDOW_LARGE_SEC, WINDOW_LARGE_SEC / 2, sfreq
)

# ── Spearman ISC at medium window (for Pearson/Spearman comparison) ──────────
print(f"  Spearman ISC ({WINDOW_MED_SEC:.0f} s / 50% overlap) …")
isc_spearman_med, _ = compute_sliding_window_isc_spearman(
    data_sw, WINDOW_MED_SEC, WINDOW_MED_SEC / 2, sfreq
)

plot_multiscale_sliding_window_isc(
    LABEL,
    isc_fine,
    times_fine,
    isc_med,
    times_med,
    isc_large,
    times_large,
    isc_spearman_med,
    sfreq,
    n_times,
    window_fine_sec=WINDOW_FINE_SEC,
    window_med_sec=WINDOW_MED_SEC,
    window_large_sec=WINDOW_LARGE_SEC,
    isc_threshold=ISC_THRESHOLD,
    n_ch_subsample=N_CH_SUBSAMPLE
    if N_CH_SUBSAMPLE is not None and N_CH_SUBSAMPLE < n_channels
    else None,
    save_path_bar=PLOTS_DIR / f"sw_isc_bar_{LABEL}.png" if SAVE_PLOTS else None,
    save_path_overlay=PLOTS_DIR / f"sw_isc_overlay_{LABEL}.png" if SAVE_PLOTS else None,
    save_path_comparison=PLOTS_DIR / f"sw_isc_pearson_vs_spearman_{LABEL}.png"
    if SAVE_PLOTS
    else None,
)

## Section 4 — Mean-Field ISC (Average Across Electrodes First)

Instead of computing ISC channel-by-channel and averaging the resulting
correlation coefficients, here the **raw signals are first averaged across
all electrodes**, producing a single global mean-field time series per subject,
and ISC is then computed on those 1-D signals.

- **LOO-ISC per subject** — each subject's mean-field signal correlated against
  the mean of all others' mean-field signals (Pearson and Spearman)
- **Pairwise ISC matrix** — subject × subject matrix using mean-field Pearson
- **Sliding-window time course** — mean-field LOO-ISC overlaid with the
  channel-average result from Section 3 to quantify agreement

**Key difference from Section 3:** Section 3 computes `mean(ISC(channel_i))`
while this section computes `ISC(mean(channel_i))`. Because correlation is
non-linear, these can differ substantially — a large discrepancy indicates
spatially heterogeneous synchrony where a small subset of channels dominates.

Uses the full electrode set (not the subsampled `data_sw`), since there is no
per-channel computation loop here.


In [ ]:
# ── Mean-field LOO-ISC (Pearson + Spearman) ───────────────────────────────
print("Computing mean-field LOO-ISC …")
loo_mf_pearson, loo_mf_spearman = compute_mean_field_loo_isc(data)
# loo_mf_pearson, loo_mf_spearman: (n_subjects,) each

plot_mean_field_loo_isc(
    {LABEL: loo_mf_pearson},
    {LABEL: loo_mf_spearman},
    save_path=PLOTS_DIR / f"mf_loo_isc_per_subject_{LABEL}.png" if SAVE_PLOTS else None,
)

In [ ]:
# ── Mean-field pairwise ISC (Pearson) ──────────────────────────────────────
print("Computing mean-field pairwise ISC …")
pair_mf = compute_mean_field_pairwise_isc(data)
# pair_mf: (n_subjects, n_subjects)

plot_mean_field_pairwise_isc(
    {LABEL: pair_mf},
    save_path=PLOTS_DIR / f"mf_pairwise_isc_{LABEL}.png" if SAVE_PLOTS else None,
)

In [ ]:
# ── Mean-field sliding-window LOO-ISC (medium window, 50% overlap) ─────────
print(
    f"Computing mean-field sliding-window ISC ({WINDOW_MED_SEC:.0f} s / 50% overlapping) …"
)
isc_mf_tc, _ = compute_mean_field_sliding_window_isc(
    data, WINDOW_MED_SEC, WINDOW_MED_SEC / 2, sfreq
)
# isc_mf_tc: (n_windows,)

plot_mean_field_vs_channel_avg_isc(
    {LABEL: isc_mf_tc},
    {LABEL: isc_med.mean(axis=1)},
    isc_threshold=ISC_THRESHOLD,
    window_sec=WINDOW_MED_SEC,
    step_sec=WINDOW_MED_SEC / 2,
    save_path=PLOTS_DIR / f"mf_vs_channel_avg_isc_{LABEL}.png" if SAVE_PLOTS else None,
)